In [1]:
import torch # core PyTorch library
import torch.nn as nn # provides neural network building blocks 
import torch.optim as optim # provides optimization algorithms (like SGD, Adam)
from torch.utils.data import DataLoader # helps load data in mini-batches
from torchvision import datasets # gives us ready-to-use datasets like MNIST
from torchvision import transforms # apply preprocessing to the images (like converting to tensors, normalizing)


In [2]:
# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
# Hyper-parameters & Device

# Parameters: the internal knowledge the model builds (the what) - weights & biases.
# Hyperparameters: the external environment and strategy you provide to guide that build (the how) - learning rate, batch size &...


# Hyperparameters: knobs that control the learning process
input_size = 28 * 28   # MNIST images are 28x28 pixels
hidden_size = 128      # number of neurons in the hidden layer
num_classes = 10       # digits 0-9
num_epochs = 5
batch_size = 64
learning_rate = 0.001


In [4]:
# Dataset & DataLoader:


# Download and load training dataset
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    # convert image to tensor and normalize to [0, 1]
    transform=transforms.ToTensor(),
    download=True
)

# Download and load test dataset
test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    # convert image to tensor and normalize to [0, 1]
    transform=transforms.ToTensor(),
    download=True
)

# Create DataLoaders to iterate over the dataset in mini-batches
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    # each epoch sees the data in a different order.
    shuffle=True
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False
)


100%|██████████| 9.91M/9.91M [00:08<00:00, 1.19MB/s]
100%|██████████| 28.9k/28.9k [00:02<00:00, 10.5kB/s]
100%|██████████| 1.65M/1.65M [00:09<00:00, 171kB/s] 
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.21MB/s]


![alt text](image.png)

In [6]:
# fully connected feedforward network:

class MLP(nn.Module):
    """
    A simple fully connected (feedforward) neural network:
    Input (784) -> Hidden (hidden_size) -> ReLU -> Output (10)
    """
    def __init__(self, input_size, hidden_size, num_classes):
        super(MLP, self).__init__()
        
        # Fully connected layer from input to hidden
        self.fc1 = nn.Linear(input_size, hidden_size)
        
        # Non-linear activation function
        self.relu = nn.ReLU()
        
        # Fully connected layer from hidden to output (logits)
        self.fc2 = nn.Linear(hidden_size, num_classes)


    def forward(self, x):
        # x shape: (batch_size, 1, 28, 28)
        
        # 1) Flatten the image into a vector of size 784
        #    .view keeps batch_size as -1 and flattens the rest
        x = x.view(x.size(0), -1)  # Flatten to (batch_size, 784)
        
        # 2) First linear layer
        x = self.fc1(x)            # (batch_size, hidden_size)
        
        # 3) Apply ReLU activation
        x = self.relu(x)
        
        # 4) Second linear layer
        x = self.fc2(x)            # (batch_size, num_classes)
        
        # The final layer produces logits (unnormalized scores). 
        # The loss function will handle the softmax internally.
        return x

# Create model instance and move it to the device (CPU or GPU)
model = MLP(input_size, hidden_size, num_classes).to(device)
print(model)


MLP(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


In [7]:
# Loss Function and Optimizer:
# Loss function: Measures how far the model’s predictions are from the true labels.
# Cross-entropy loss is commonly used for multi-class classification.
# Optimizer: Updates the model parameters (weights) to minimize the loss using gradients.
# model.parameters() passes all trainable parameters of the model to the optimizer.


# Cross entropy combines log-softmax and negative log likelihood loss.
criterion = nn.CrossEntropyLoss()

# Adam is a popular variant of stochastic gradient descent.
optimizer = optim.Adam(model.parameters(), lr=learning_rate)



In [ ]:
# Training Loop:

def train(model, train_loader, criterion, optimizer, num_epochs, device):
    model.train()  #  Enables layers like Dropout (randomly shuts off neurons) and Batch Normalization (tracks running averages of mean/variance).
    for epoch in range(num_epochs):
        running_loss = 0.0
        
        for batch_idx, (images, labels) in enumerate(train_loader):
            # Move data to the device
            images = images.to(device)
            labels = labels.to(device)
            
            # 1) Forward pass: compute predictions
            outputs = model(images)
            
            # 2) Compute loss
            loss = criterion(outputs, labels)
            
            # 3) Backward pass: compute gradients
            optimizer.zero_grad()  # must be called before backward() to avoid accumulating gradients from previous batches
            loss.backward()        # Uses backpropagation to compute gradients of the loss w.r.t. each parameter.
            
            # 4) Adjusts the parameters slightly to reduce loss
            optimizer.step()
            
            running_loss += loss.item()
        
        # Print average loss over the epoch
        avg_loss = running_loss / len(train_loader)
        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {avg_loss:.4f}")


In [ ]:
# Evaluation Loop(Test Accuracy)

def evaluate(model, test_loader, device):
    model.eval()  # Dropout is deactivated (it lets all data through), and Batch Normalization uses the 
                  # statistics it learned during training instead of the current batch
    correct = 0
    total = 0
    
    # disables gradient computation, which: Saves memory & Speeds up evaluation
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)  # (batch_size, num_classes)
            
            # Get predicted class by taking the index of the max logit:
            # returns the maximum along the class dimension and its index; we use the index as the predicted class.
            _, predicted = torch.max(outputs, dim=1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100.0 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy


In [10]:
# --- SAVE / LOAD LOGIC ---

def save_model(model: nn.Module, path: str = "model.pth"):
    # state_dict is a dictionary mapping each layer to its parameter tensors
    torch.save(model.state_dict(), path)
    print(f"Model weights saved to {path}")

def load_model(model: nn.Module, path: str = "model.pth"):
    # We load the dictionary and then inject it into the model object
    model.load_state_dict(torch.load(path))
    model.eval() # Always set to eval after loading
    print(f"Model weights loaded from {path}")


In [13]:
if __name__ == "__main__":
    # train(model, train_loader, criterion, optimizer, num_epochs, device)
    # evaluate(model, test_loader, device)
    # save_model(model)

    load_model(model)


Model weights loaded from model.pth


In [15]:
# --- INSPECTING TENSORS ---

# Let's see what a weight tensor looks like internally
# model.layers[0].weight is a Tensor
model.fc1.weight
# Shape: [hidden_size, input_size]
# Memory: (hidden_size * input_size) * 4 bytes (for float32)

Parameter containing:
tensor([[-0.0024,  0.0178, -0.0327,  ..., -0.0182,  0.0192,  0.0283],
        [ 0.0229,  0.0035, -0.0131,  ..., -0.0018, -0.0017,  0.0142],
        [-0.0287, -0.0086,  0.0163,  ..., -0.0171,  0.0182,  0.0045],
        ...,
        [ 0.0112, -0.0251, -0.0232,  ..., -0.0022,  0.0119,  0.0343],
        [ 0.0281,  0.0316,  0.0086,  ..., -0.0228,  0.0117, -0.0248],
        [ 0.0248,  0.0237,  0.0117,  ...,  0.0303, -0.0344,  0.0084]],
       device='cuda:0', requires_grad=True)

In [ ]:
# gives details of your system's resources


import platform
import psutil # You might need to: pip install psutil

def print_system_diagnostics():
    print("="*30)
    print(" SYSTEM & HARDWARE INFO ")
    print("="*30)
    
    # 1. Basic OS/CPU Info
    print(f"OS:       {platform.system()} {platform.release()}")
    print(f"CPU:      {platform.processor()}")
    print(f"RAM:      {psutil.virtual_memory().total / 1024**3:.2f} GB total")
    
    # 2. CUDA / GPU Info
    if torch.cuda.is_available():
        device_id = torch.cuda.current_device()
        props = torch.cuda.get_device_properties(device_id)
        
        # Hardware Stats
        total_vram = props.total_memory / 1024**2 # Convert to MB
        
        # PyTorch Internal Memory Stats
        # 'Allocated' is what your tensors actually occupy
        allocated = torch.cuda.memory_allocated(device_id) / 1024**2
        # 'Reserved' is the "cache" PyTorch has grabbed from the OS
        reserved = torch.cuda.memory_reserved(device_id) / 1024**2
        # 'Free' inside the cache
        free_inside_cache = reserved - allocated
        
        print(f"GPU:      {props.name}")
        print(f"CUDA:     {torch.version.cuda} (Capability {props.major}.{props.minor})")
        print(f"MultiProcessors: {props.multi_processor_count}")
        print("-" * 30)
        print(f"Total VRAM Capacity:  {total_vram:.2f} MB")
        print(f"Reserved by PyTorch:  {reserved:.2f} MB ({(reserved/total_vram)*100:.1f}%)")
        print(f"  └─ Actually Used:   {allocated:.2f} MB")
        print(f"  └─ Empty Cache:     {free_inside_cache:.2f} MB")
        
        # Real-time estimate of what's left for other apps
        # Note: Total - Reserved is what the OS/Other apps still have access to
        external_free = total_vram - reserved
        print(f"VRAM available to OS: {external_free:.2f} MB")
        
    else:
        print("CUDA is NOT available on this machine.")
    print("="*30)

print_system_diagnostics()


 SYSTEM & HARDWARE INFO 
OS:       Windows 11
CPU:      Intel64 Family 6 Model 165 Stepping 2, GenuineIntel
RAM:      15.77 GB total
GPU:      NVIDIA GeForce GTX 1660 Ti
CUDA:     12.9 (Capability 7.5)
MultiProcessors: 24
------------------------------
Total VRAM Capacity:  6143.69 MB
Reserved by PyTorch:  24.00 MB (0.4%)
  └─ Actually Used:   17.80 MB
  └─ Empty Cache:     6.20 MB
VRAM available to OS: 6119.69 MB
